# Feature engineering - advanced data preparation pipeline  | Sebislaw

## Libraries

In [1]:
from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd
from pandas.plotting import scatter_matrix

import matplotlib.pyplot as plt
import plotly.express as px
from pandas.plotting import parallel_coordinates
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold, RandomizedSearchCV
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.neural_network import MLPClassifier

import xgboost as xgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
# from tabpfn import TabPFNClassifier

## Data

In [2]:
data_path = '..\\..\\..\\data'
pd.set_option('display.max_columns', None)

# The Basics ------------------------------------------------------------------------
# Men
MTeams = pd.read_csv(join(data_path, 'MTeams.csv'))
MSeasons = pd.read_csv(join(data_path, 'MSeasons.csv'))
MNCAATourneySeeds = pd.read_csv(join(data_path, 'MNCAATourneySeeds.csv'))
MRegularSeasonCompactResults = pd.read_csv(join(data_path, 'MRegularSeasonCompactResults.csv'))
MNCAATourneyCompactResults = pd.read_csv(join(data_path, 'MNCAATourneyCompactResults.csv'))
# Women
WTeams = pd.read_csv(join(data_path, 'WTeams.csv'))
WSeasons = pd.read_csv(join(data_path, 'WSeasons.csv'))
WNCAATourneySeeds = pd.read_csv(join(data_path, 'WNCAATourneySeeds.csv'))
WRegularSeasonCompactResults = pd.read_csv(join(data_path, 'WRegularSeasonCompactResults.csv'))
WNCAATourneyCompactResults = pd.read_csv(join(data_path, 'WNCAATourneyCompactResults.csv'))
# Other
SampleSubmissionStage1 = pd.read_csv(join(data_path, 'SampleSubmissionStage1.csv'))
SampleSubmissionStage2 = pd.read_csv(join(data_path, 'SampleSubmissionStage2.csv'))
SeedBenchmarkStage1 = pd.read_csv(join(data_path, 'SeedBenchmarkStage1.csv'))

# Team Box Scores ------------------------------------------------------------------------
# Men
MRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'MRegularSeasonDetailedResults.csv'))
MNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'MNCAATourneyDetailedResults.csv'))
# Women
WRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'WRegularSeasonDetailedResults.csv'))
WNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'WNCAATourneyDetailedResults.csv'))

# Geography ------------------------------------------------------------------------
# All
Cities = pd.read_csv(join(data_path, 'Cities.csv'))
Conferences = pd.read_csv(join(data_path, 'Conferences.csv'))
# Men
MGameCities = pd.read_csv(join(data_path, 'MGameCities.csv'))
# Women
WGameCities = pd.read_csv(join(data_path, 'WGameCities.csv'))

# Public Rankings ------------------------------------------------------------------------
# Men
MMasseyOrdinals = pd.read_csv(join(data_path, 'MMasseyOrdinals.csv')) # men only

# Supplements ------------------------------------------------------------------------
# Men
MTeamCoaches = pd.read_csv(join(data_path, 'MTeamCoaches.csv')) # men only
MTeamConferences = pd.read_csv(join(data_path, 'MTeamConferences.csv'))
MConferenceTourneyGames = pd.read_csv(join(data_path, 'MConferenceTourneyGames.csv'))
MSecondaryTourneyTeams = pd.read_csv(join(data_path, 'MSecondaryTourneyTeams.csv'))
MSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'MSecondaryTourneyCompactResults.csv'))
MTeamSpellings = pd.read_csv(join(data_path, "MTeamSpellings.csv"), encoding='cp1252')
MNCAATourneySlots = pd.read_csv(join(data_path, 'MNCAATourneySlots.csv'))
MNCAATourneySeedRoundSlots = pd.read_csv(join(data_path, 'MNCAATourneySeedRoundSlots.csv')) # men only
# Women
WTeamConferences = pd.read_csv(join(data_path, 'WTeamConferences.csv'))
WConferenceTourneyGames = pd.read_csv(join(data_path, 'WConferenceTourneyGames.csv'))
WSecondaryTourneyTeams = pd.read_csv(join(data_path, 'WSecondaryTourneyTeams.csv'))
WSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'WSecondaryTourneyCompactResults.csv'))
WTeamSpellings = pd.read_csv(join(data_path, 'WTeamSpellings.csv'), encoding='cp1252')
WNCAATourneySlots = pd.read_csv(join(data_path, 'WNCAATourneySlots.csv'))

## Data preparation pipeline

In [3]:
def prepare_data(df):
        
    """
    This function duplicates and flips a game record.
    Now two records with the same data are present, 
    but viewed from perspectives of two different teams.
    """
    
    df = df[[
         'Season', 'DayNum', 'NumOT',
         'WTeamID',  'WScore', 'WLoc',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
         'LTeamID', 'LScore',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'
    ]]
    dfswap = df[[
         'Season', 'DayNum', 'NumOT',
         'LTeamID', 'LScore', 'WLoc',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF',
         'WTeamID',  'WScore',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
    ]].copy()
    
    dfswap.loc[df['WLoc'] == 'H', 'WLoc'] = 'A'
    dfswap.loc[df['WLoc'] == 'A', 'WLoc'] = 'H'
        
    df = df.rename(columns={'WLoc': 'location'})
    dfswap = dfswap.rename(columns={'WLoc': 'location'})
        
    df.columns = [x.replace('W','T1_').replace('L','T2_') for x in list(df.columns)]
    dfswap.columns = [x.replace('L','T1_').replace('W','T2_') for x in list(dfswap.columns)]
    
    output = pd.concat([df, dfswap]).reset_index(drop=True)
    output.loc[output.location=='N','location'] = '0'
    output.loc[output.location=='H','location'] = '1'
    output.loc[output.location=='A','location'] = '-1'
    output.location = output.location.astype(int)
        
    output['PointDiff'] = output['T1_Score'] - output['T2_Score']
    
    return output

def get_data(regular_results, tourney_results, seeds, prepared=False, location_multiplier=[1, 1], win_ratio_days_back=14):

    """
    This function uses the prepare_data function in order to create
    a data frame with season statistics for each team.
    These statistics are added to records with games played in
    tournament to make data 'x' used in model to predict the game 
    result 'y'. The output is a data frame that contains data 'x'
    and also label 'y' can be easily calculated based on score difference in matches.
    """

    if prepared:
        regular_data = regular_results.copy()
        tourney_data = tourney_results.copy()
    else:
        # make data frames with extra rows to represent the perspective of losing team
        regular_data = prepare_data(regular_results)
        tourney_data = prepare_data(tourney_results)

    # ----------------------------------- Add reward/penalty for playing in home or away
    if location_multiplier[0] == 1 and location_multiplier[1] == 1:
        # data frame with mean game statistics for a given team in a given season
        season_statistics  = regular_data.groupby(["Season", 'T1_TeamID'])[
            [
                'T1_Score', 'T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA',
                'T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF',
                'T2_Score', 'T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA',
                'T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF',
                'PointDiff'
            ]
        ].agg('mean').reset_index()
    else:
        # Define which columns to adjust (you can add or remove columns as needed)
        T1_cols = ['T1_Score','T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA','T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF']
        T2_cols = ['T2_Score','T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA','T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF']
    
        # Convert the relevant columns to float before applying the adjustment function.
        cols_to_float = T1_cols + T2_cols
        regular_data[cols_to_float] = regular_data[cols_to_float].astype(float)
        
        def adjust_stats(row):
            # Determine multipliers based on location
            if row['location'] == 1:
                factor_T1 = location_multiplier[0]  # penalize Team1 stats (home)
                factor_T2 = location_multiplier[1]  # boost Team2 stats
            elif row['location'] == -1:
                factor_T1 = location_multiplier[1]  # boost Team1 stats (away)
                factor_T2 = location_multiplier[0]  # penalize Team2 stats
            else:
                factor_T1 = 1.0
                factor_T2 = 1.0
        
            # Adjust Team1 stats
            for col in T1_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T1
        
            # Adjust Team2 stats
            for col in T2_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T2
        
            # Recalculate derived statistics (if needed)
            if 'T1_Score' in row and 'T2_Score' in row:
                row['PointDiff'] = row['T1_Score'] - row['T2_Score']
            return row
        
        # Apply the adjustment function row-wise.
        regular_data_adjusted = regular_data.apply(adjust_stats, axis=1)
        
        # Now group by Season and T1_TeamID to compute season averages for the adjusted statistics.
        stats_columns = T1_cols[1:] + T2_cols[1:] + ['PointDiff']  # Exclude T1_TeamID from stats if present.
        season_statistics = regular_data_adjusted.groupby(["Season", 'T1_TeamID'])[stats_columns].agg('mean').reset_index()
    # -----------------------------------
    
    # mean statistics for team and team's opponent's
    season_statistics_T1 = season_statistics.copy()
    season_statistics_T2 = season_statistics.copy()
    
    season_statistics_T1.columns = ["T1_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T1.columns)]
    season_statistics_T2.columns = ["T2_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T2.columns)]
    season_statistics_T1.columns.values[0] = "Season"
    season_statistics_T2.columns.values[0] = "Season"
    season_statistics_T1 = season_statistics_T1.rename(columns={'T1_Score': 'T1_Score_mean'})
    season_statistics_T2 = season_statistics_T2.rename(columns={'T2_Score': 'T2_Score_mean'})
    
#     print('season_statistics_T2 in get data')
#     print(season_statistics_T2)
    
    # data frame containing game's result
#     print('tourney_data BEFORE MERGE')
#     print(tourney_data)
    tourney_data = tourney_data[['Season', 'DayNum', 'T1_TeamID', 'T1_Score', 'T2_TeamID' ,'T2_Score', 'location']]
    tourney_data = pd.merge(tourney_data, season_statistics_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, season_statistics_T2, on = ['Season', 'T2_TeamID'], how = 'left')
#     print('tourney_data AFTER MERGE NO NA')
#     print(tourney_data[tourney_data['T1_FGM'].notna()])

    calculate_win_ratio_days_back = 132 - win_ratio_days_back
    
    # data frame with win fraction from last x days for a given team in a given season
    last14days_stats_T1 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T1['win'] = np.where(last14days_stats_T1['PointDiff']>0,1,0)
    last14days_stats_T1 = last14days_stats_T1.groupby(['Season','T1_TeamID'])['win'].mean().reset_index(name='T1_win_ratio_14d')
    
    last14days_stats_T2 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T2['win'] = np.where(last14days_stats_T2['PointDiff']<0,1,0)
    last14days_stats_T2 = last14days_stats_T2.groupby(['Season','T2_TeamID'])['win'].mean().reset_index(name='T2_win_ratio_14d')
    
    # add to tourney_data column with win fraction for winning and losing team
    tourney_data = pd.merge(tourney_data, last14days_stats_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, last14days_stats_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # get seeds with no regional division
    seeds['seed'] = seeds['Seed'].apply(lambda x: int(x[1:3]))
    
    # give each team a raw seed
    seeds_T1 = seeds[['Season','TeamID','seed']].copy()
    seeds_T2 = seeds[['Season','TeamID','seed']].copy()
    seeds_T1.columns = ['Season','T1_TeamID','T1_seed']
    seeds_T2.columns = ['Season','T2_TeamID','T2_seed']
    
    # add seeds to turney data for team 1 and team 2
    tourney_data = pd.merge(tourney_data, seeds_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, seeds_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # add a seed difference column
    tourney_data["Seed_diff"] = tourney_data["T1_seed"] - tourney_data["T2_seed"]

    return tourney_data

def get_df(seeds,
            season_games, season_range, days_back,
            tourney_games, tourney_range, 
            location_multiplier=[1, 1],
           win_ratio_days_back=14):

    """
    This function uses get_data function to get a data
    frame which is then used to make 'x' and 'y' data
    used in models.
    """
    
    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]
    
    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Get final data frame
    df = get_data(regular_results, tourney_results, seeds, location_multiplier=[1, 1], win_ratio_days_back=win_ratio_days_back)
    
    return df

def get_final_df(seeds,
                   season_games, season_range, days_back,
                   tourney_games, tourney_range, 
                   SampleSubmissionStage2,
                  location_multiplier=[1, 1],
                win_ratio_days_back=14):

    """
    This function works the same as function get_x_y,
    but also creates (at the moment it's the same as get_x_y)
    aditional data points mostly with NaN values that matches
    the submission format (parsed team matchups with the sample submission file).
    """

    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]

    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Assume sample_submission is a DataFrame with an "ID" column like "2023_1101_1102"
    # and tourney_results is a DataFrame with columns including: Season, WTeamID, LTeamID, DayNum, WScore, LScore, WLoc, etc.
    # Filter rows where the ID starts with the specified season (followed by an underscore)
    final_season = tourney_range[0]
    sample_submission_copy = SampleSubmissionStage2.copy()
    sample_submission = sample_submission_copy[sample_submission_copy['ID'].str.startswith(f"{final_season}_")]
    sample_submission = sample_submission.drop(columns=['Pred'])
    
    sample_submission[['Season', 'Team1', 'Team2']] = sample_submission['ID'].str.split('_', expand=True)
    sample_submission['Season'] = sample_submission['Season'].astype(float)
    sample_submission['Team1'] = sample_submission['Team1'].astype(float)
    sample_submission['Team2'] = sample_submission['Team2'].astype(float)
    
    regular_data_final = prepare_data(regular_results)
    tourney_data_final  = prepare_data(tourney_results)
    
#     print("-------------")
#     print(regular_data_final)
    
#     print(list(tourney_data_final))
    sample_submission = sample_submission.rename(columns={'Team1': 'T1_TeamID', 'Team2': 'T2_TeamID'})
#     print('list(sample_submission)')
#     print(list(sample_submission))
    tourney_data_final = pd.merge(
        sample_submission,
        tourney_data_final,
        on=['Season', 'T1_TeamID', 'T2_TeamID'],
#         left_on=['Season', 'Team1', 'Team2'],
#         right_on=['Season', 'T1_TeamID', 'T2_TeamID'],
        how='left'
    )
#     tourney_data_final.rename(columns={'Team1': 'T1_TeamID', 'Team2': 'T2_TeamID'}, inplace=True)
#     print("TESM AIN")
#     print(tourney_data_final['T1_TeamID'])
    tourney_data_final = tourney_data_final[['Season', 'DayNum', 'NumOT', 'T1_TeamID', 'T1_Score', 'location', 'T1_FGM', 'T1_FGA', 'T1_FGM3', 'T1_FGA3', 'T1_FTM', 'T1_FTA', 'T1_OR', 'T1_DR', 'T1_Ast', 'T1_TO', 'T1_Stl', 'T1_Blk', 'T1_PF', 'T2_TeamID', 'T2_Score', 'T2_FGM', 'T2_FGA', 'T2_FGM3', 'T2_FGA3', 'T2_FTM', 'T2_FTA', 'T2_OR', 'T2_DR', 'T2_Ast', 'T2_TO', 'T2_Stl', 'T2_Blk', 'T2_PF', 'PointDiff']]
#     print(tourney_data_final['T1_TeamID'])
#     print("-------------")
    
    tourney_data =  get_data(regular_data_final, tourney_data_final,
                             seeds, prepared=True,
                          win_ratio_days_back=win_ratio_days_back)
#     print("NOTNA")
#     print(tourney_data_final['T1_TeamID'])
#     print(tourney_data[tourney_data['T1_FGM'].notna()])
    tourney_data = pd.merge(
        sample_submission,
        tourney_data,
        on=['Season', 'T1_TeamID', 'T2_TeamID'],
#         left_on=['Season', 'Team1', 'Team2'],
#         right_on=['Season', 'T1_TeamID', 'T2_TeamID'],
        how='left'
    )
#     tourney_data['T1_TeamID'] = tourney_data['Team1']
#     tourney_data['T2_TeamID'] = tourney_data['Team2']
#     tourney_data = tourney_data.drop(['ID', 'Team1', 'Team2'], axis=1)
    tourney_data = tourney_data.drop(['ID'], axis=1)
    
    return tourney_data

def correct_predictions_based_on_seed(x, y, maximum_favoured_seed = 4, number_of_added_columns=1):
    """
    Sets the winnning chance to 1 or 0 based on the seed difference.
    """
    for i in range(len(x)):
        if x[i][-1-number_of_added_columns] <= (-16 + maximum_favoured_seed * 2 - 1):
            y[i] = 1
        elif x[i][-1-number_of_added_columns] >= (16 - maximum_favoured_seed * 2 + 1):
            y[i] = 0
    return y

def clear_na_from_x_y(x, y):
    """
    The data frame for final season is in format matching the submission file.
    This function clears NaNs from data.
    """
    # Create masks for training data:
    mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
    x_clean = x[mask_train]
    y_clean = y[mask_train]
    return x_clean, y_clean

def x_y_from_data_frame(df):
    # Prepare data and labels
    x = df[list(df.columns[7:])].values
    y = np.where(
        df[['T1_Score', 'T2_Score']].isnull().any(axis=1),
        np.nan,
        np.where(df['T1_Score'] - df['T2_Score'] > 0, 1, 0)
    )
    return x, y

def get_all_core_data(
    regular_results, tourney_results, seeds, SampleSubmissionStage2,
    final_season = 2024, # the season we want to predict, so for out submission it will be 2025
    start_season = 2005, # from which ponit should we begin creating data
    season_years_list  = [[i-1, i] for i in range(2005, 2024+1)], # at which seasons to look at when calculating team's stats
    days_back = 15, # how many days back from the start of tourney to calculate team's stats per season
    maximum_favoured_seed = 4, # Set the predicted probability of winning to 1 for seeds <= maximum_favoured_seed and to 0 for >= 16-maximum_favoured_seed
    location_multiplier=[0.95, 1.05], # home penalty, away bonus 
    include_men = True, # include M... data sets when preparing x and y
    include_women = True, # include W... data sets when preparing x and y
    win_ratio_days_back = 14):

    """
    The idea of this function is to easily get data needed to train and test the model later on, with minimal code
    to not clutter the netebook.
    
    This function outputs df, x, y, df_final, x_final_season, y_final_season.
    
    df is a data frame with first 6 columns from tourney games and other calculated from other data frames.
    
    Adding a column to df and executing x_y_from_data_frame(df) function will yield x with added data.
    
    Note that df_final has the same structure as df, but also with rows with NaNs. The rows with missing information are there
    to match the sumbission file format. The separation of those data frames is to ensure that information from last season doesn't
    leak into training data due to poorly written code.
    
    x has df columns from location onwardsthe columns before that are from tourney games and are used to calculate y (based on points).
    
    y has label 0 or 1 (lose or win) and nan if the correspoinding data in x was nan.
    
    There is also x_final_season and y_final_season aquired from df_final, which are the same as x and y, but like df_final, they have NaNs. 
    """

    # This ensures that we simulate the scenario in competition
    tourney_results_final = tourney_results[tourney_results['Season'] == final_season]
    tourney_results = tourney_results[tourney_results['Season'] < final_season]
    regular_results = regular_results[regular_results['Season'] != 2020] # This year had no tournament data
    tourney_years_list = [[i] for i in range(start_season, final_season+1)]
    
    # Arrays to store data
    data = []
    data_final = []
    for season_years, tourney_years in zip(season_years_list, tourney_years_list):
            
        # Create data separately for the of games
        # The separation is to ensure there is no data leak
        if tourney_years[0] == final_season:
            data_tmp = get_final_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results_final, tourney_years,
                SampleSubmissionStage2=SampleSubmissionStage2,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data_final.append(data_tmp)
        else:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data.append(data_tmp)
            
    df = pd.concat(data, ignore_index=True)
    df_final = pd.concat(data_final, ignore_index=True)
    
    return df, df_final

def brier_for_all_years(year_range, season_years_list):
    
    df_train_women_list = []
    df_test_women_list = []
    df_train_men_list = []
    df_test_men_list = []
    
    for year, season_years in zip(year_range, season_years_list):
    
        final_season = year

        for sex in ['woman', 'man']:
            
            if sex == 'woman':
                include_men = False
                include_women = True
            elif sex == 'man':
                include_men = True
                include_women = False

            # ----------------------------------------------------------
            # READ DATA
            regular_results = pd.concat([
                MRegularSeasonDetailedResults.copy() if include_men else None,
                WRegularSeasonDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            tourney_results = pd.concat([
                MNCAATourneyDetailedResults.copy() if include_men else None,
                WNCAATourneyDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            seeds = pd.concat([
                MNCAATourneySeeds.copy() if include_men else None,
                WNCAATourneySeeds.copy() if include_women else None
            ], ignore_index=True)
            # ----------------------------------------------------------
            # GET ALL DATA NEEDED TO USE THE MODELS
            df_train, df_test = get_all_core_data(
                regular_results, tourney_results, seeds, SampleSubmissionStage2, final_season = final_season,
                start_season = start_season, season_years_list  = season_years, days_back = days_back,
                maximum_favoured_seed = maximum_favoured_seed, location_multiplier=location_multiplier,
                include_men = include_men, include_women = include_women, win_ratio_days_back = win_ratio_days_back)
            # ----------------------------------------------------------
            # ADD TEAM AND COACH ELO AND REPLACE NAN WITH MEAN
            def add_elo_columns_team1(df):
                df = df.copy()
                df = pd.merge(
                    df,
                    elo[['Season', 'TeamID', 'TeamELO', 'CoachELO']],
                    left_on=['Season', 'T1_TeamID'],
                    right_on=['Season', 'TeamID'],
                    how='left'
                )
                df = df.drop(['TeamID'], axis=1)
                df = df.rename(columns={'TeamELO': 'T1_TeamELO', 'CoachELO': 'T1_CoachELO'})
                return df

            def add_elo_columns_team2(df):
                df = df.copy()
                df = pd.merge(
                    df,
                    elo[['Season', 'TeamID', 'TeamELO', 'CoachELO']],
                    left_on=['Season', 'T2_TeamID'],
                    right_on=['Season', 'TeamID'],
                    how='left'
                )
                df = df.drop(['TeamID'], axis=1)
                df = df.rename(columns={'TeamELO': 'T2_TeamELO', 'CoachELO': 'T2_CoachELO'})
                return df
            df_train = add_elo_columns_team1(df_train)
            df_train = add_elo_columns_team2(df_train)
            df_test  = add_elo_columns_team1(df_test)
            df_test  = add_elo_columns_team2(df_test)
            # ----------------------------------------------------------
            if sex == 'woman':
                df_train_women_list.append(df_train.copy())
                df_test_women_list.append(df_test.copy())
            elif sex == 'man':
                df_train_men_list.append(df_train.copy())
                df_test_men_list.append(df_test.copy())
                
        for i in range(len(df_train_men_list)):
            df_train_women_list[i] = df_train_women_list[i][list(df_train_men_list[0])]
            df_test_women_list[i] = df_test_women_list[i][list(df_train_men_list[0])]
            
    return df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list

In [4]:
elo = pd.read_csv(join(data_path, 'elo.csv'))
elo['CoachELO'] = elo['CoachELO'].fillna(elo['CoachELO'].mean())
elo = elo.drop(['CoachName'], axis=1)
elo = elo.loc[elo.groupby(['Season', 'TeamID'])['DayNum'].idxmax()]
elo

,Season,DayNum,TeamID,TeamELO,CoachELO
7369,1985,127,1102,1384.975555,1386.121212
6759,1985,119,1103,1456.386146,1458.732587
7735,1985,144,1104,1662.447995,1663.573415
7284,1985,126,1106,1435.719069,1435.498557
7609,1985,131,1108,1642.627949,1643.394432
...,...,...,...,...,...
674034,2025,117,3476,1137.756460,1612.000528
673887,2025,117,3477,1135.019364,1612.000528
674035,2025,117,3478,1209.409031,1612.000528
673261,2025,115,3479,1241.658681,1612.000528


## Get data frame

In [5]:
columns_to_include_women = [
 'Season',
#  'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'T1_TeamELO',
'T2_TeamELO',
    'T1_CoachELO',
    'T2_CoachELO'
]
columns_to_include_men = columns_to_include_women

year_range = [2025]
start_season = 2010 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 25 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.95, 1.05] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
maximum_favoured_seed = 0
# -------------------------------------------
df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)
x_train_women_list = []
x_test_women_list = []
x_train_men_list = []
x_test_men_list = []
y_train_women_list = []
y_test_women_list = []
y_train_men_list = []
y_test_men_list = []
for i in range(len(year_range)):

    if len(year_range) > 1:
        df_train_women_list[i] = df_train_women_list[i][columns_to_include_women]
        df_test_women_list[i] = df_test_women_list[i][columns_to_include_women]
        df_train_men_list[i] = df_train_men_list[i][columns_to_include_men]
        df_test_men_list[i] = df_test_men_list[i][columns_to_include_men]
        
        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list[i])
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list[i])
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list[i])
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list[i])
        
        x_train_women_list.append(x_train_women)
        x_test_women_list.append(x_test_women)
        x_train_men_list.append(x_train_men)
        x_test_men_list.append(x_test_men)

        y_train_women_list.append(y_train_women)
        y_test_women_list.append(y_test_women)
        y_train_men_list.append(y_train_men)
        y_test_men_list.append(y_test_men)
    
    else:
        df_train_women_list = df_train_women_list[0][columns_to_include_women]
        df_test_women_list = df_test_women_list[0][columns_to_include_women]
        df_train_men_list = df_train_men_list[0][columns_to_include_men]
        df_test_men_list = df_test_men_list[0][columns_to_include_men]

        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list)
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list)
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list)
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list)

In [6]:
df_train_women_list

,Season,T1_TeamID,T1_Score,T2_TeamID,T2_Score,location,T1_FGM,T1_FGA,T1_FGM3,T1_FGA3,T1_OR,T1_Ast,T1_TO,T1_Stl,T1_PF,T1_opponent_FGM,T1_opponent_FGA,T1_opponent_FGM3,T1_opponent_FGA3,T1_opponent_OR,T1_opponent_Ast,T1_opponent_TO,T1_opponent_Stl,T1_opponent_PF,T1_PointDiff,T2_FGM,T2_FGA,T2_FGM3,T2_FGA3,T2_OR,T2_Ast,T2_TO,T2_Stl,T2_PF,T2_opponent_FGM,T2_opponent_FGA,T2_opponent_FGM3,T2_opponent_FGA3,T2_opponent_OR,T2_opponent_Ast,T2_opponent_TO,T2_opponent_Stl,T2_opponent_PF,T2_PointDiff,T1_win_ratio_14d,T2_win_ratio_14d,T1_seed,T2_seed,Seed_diff,T1_TeamELO,T2_TeamELO,T1_CoachELO,T2_CoachELO
0,2010,3124,69,3201,55,0,22.857143,54.571429,2.285714,9.285714,11.142857,13.571429,13.857143,5.428571,14.571429,23.285714,59.857143,4.428571,14.428571,11.142857,10.714286,13.285714,6.857143,19.142857,4.428571,27.571429,66.142857,10.285714,26.428571,14.571429,14.428571,15.428571,10.857143,17.571429,25.285714,60.285714,4.000000,13.285714,13.285714,13.428571,20.857143,7.000000,16.000000,16.142857,0.500000,0.750000,4,13,-9,2110.009681,1784.461585,1612.000528,1612.000528
1,2010,3173,67,3395,66,0,24.600000,57.200000,4.800000,16.400000,13.800000,14.400000,15.000000,6.200000,18.000000,21.000000,60.400000,5.000000,14.600000,15.400000,9.600000,15.400000,6.600000,18.800000,11.400000,23.600000,58.200000,7.000000,20.200000,9.600000,15.400000,14.600000,8.000000,17.200000,22.600000,57.600000,6.000000,19.600000,13.000000,14.800000,19.000000,8.200000,16.200000,4.400000,0.500000,0.333333,8,9,-1,1820.315166,1829.517391,1612.000528,1612.000528
2,2010,3181,72,3214,37,1,24.166667,61.666667,5.833333,15.333333,17.500000,12.833333,17.166667,13.500000,19.166667,20.000000,52.833333,5.333333,16.333333,15.000000,10.000000,23.666667,8.500000,21.166667,7.333333,20.285714,52.000000,3.857143,14.714286,10.000000,11.000000,14.857143,8.000000,16.857143,16.428571,45.000000,1.428571,6.428571,10.714286,6.000000,21.285714,6.285714,16.142857,11.571429,1.000000,1.000000,2,15,-13,2233.724911,1455.655863,1612.000528,1612.000528
3,2010,3199,75,3256,61,1,25.500000,59.250000,7.750000,18.750000,14.500000,12.000000,16.750000,8.250000,17.000000,22.000000,56.000000,6.500000,19.500000,10.750000,10.500000,18.750000,7.500000,19.500000,13.000000,27.750000,63.625000,4.125000,12.875000,14.500000,14.625000,14.375000,7.125000,15.500000,25.500000,64.625000,5.500000,19.125000,14.375000,12.750000,15.875000,6.750000,17.750000,4.750000,0.000000,0.800000,3,14,-11,2086.754069,1689.359916,1612.000528,1612.000528
4,2010,3207,62,3265,42,0,24.000000,62.333333,7.333333,21.666667,16.833333,16.333333,14.166667,12.000000,15.166667,20.166667,49.666667,5.833333,21.666667,10.500000,14.166667,20.166667,5.833333,14.000000,8.333333,24.333333,55.333333,6.000000,16.000000,9.666667,13.333333,11.666667,8.500000,11.500000,20.166667,58.666667,5.833333,20.833333,14.166667,9.833333,14.500000,6.500000,16.166667,12.666667,0.500000,1.000000,5,12,-7,1913.531437,1815.353018,1612.000528,1612.000528
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1783,2024,3425,73,3163,80,1,26.833333,67.333333,5.666667,19.500000,13.166667,13.500000,13.166667,7.166667,17.833333,24.666667,62.666667,6.333333,18.833333,6.833333,15.666667,12.333333,8.166667,19.666667,6.166667,27.833333,57.666667,7.333333,21.500000,6.166667,17.333333,11.500000,10.166667,11.833333,17.833333,56.166667,4.666667,20.666667,6.833333,10.666667,15.833333,5.166667,16.500000,29.833333,1.000000,1.000000,1,3,-2,2156.553539,2337.905869,1612.000528,1612.000528
1784,2024,3261,87,3234,94,-1,27.833333,68.000000,4.500000,13.166667,13.500000,13.666667,13.000000,8.166667,13.666667,23.166667,63.833333,4.833333,16.000000,9.000000,12.500000,15.500000,6.833333,17.833333,15.500000,34.666667,67.833333,14.833333,34.500000,9.666667,25.666667,13.833333,7.333333,13.333333,27.666667,70.000000,9.833333,27.500000,9.833333,17.000000,13.333333

In [7]:
df_train_men_list

,Season,T1_TeamID,T1_Score,T2_TeamID,T2_Score,location,T1_FGM,T1_FGA,T1_FGM3,T1_FGA3,T1_OR,T1_Ast,T1_TO,T1_Stl,T1_PF,T1_opponent_FGM,T1_opponent_FGA,T1_opponent_FGM3,T1_opponent_FGA3,T1_opponent_OR,T1_opponent_Ast,T1_opponent_TO,T1_opponent_Stl,T1_opponent_PF,T1_PointDiff,T2_FGM,T2_FGA,T2_FGM3,T2_FGA3,T2_OR,T2_Ast,T2_TO,T2_Stl,T2_PF,T2_opponent_FGM,T2_opponent_FGA,T2_opponent_FGM3,T2_opponent_FGA3,T2_opponent_OR,T2_opponent_Ast,T2_opponent_TO,T2_opponent_Stl,T2_opponent_PF,T2_PointDiff,T1_win_ratio_14d,T2_win_ratio_14d,T1_seed,T2_seed,Seed_diff,T1_TeamELO,T2_TeamELO,T1_CoachELO,T2_CoachELO
0,2010,1115,61,1457,44,0,18.750000,47.500000,4.375000,14.875000,11.375000,12.500000,16.125000,6.625000,23.375000,16.625000,48.875000,3.750000,13.625000,9.625000,9.125000,13.625000,8.125000,22.375000,6.000000,22.857143,57.571429,4.000000,14.714286,11.571429,11.714286,11.428571,8.000000,19.000000,21.714286,53.714286,4.428571,17.857143,12.142857,11.142857,15.000000,5.714286,17.000000,1.428571,0.800000,1.000000,16,16,0,1315.830857,1435.940191,1547.449749,1577.304377
1,2010,1124,68,1358,59,0,27.571429,55.857143,6.571429,15.857143,11.000000,13.000000,13.714286,7.142857,18.857143,24.714286,59.571429,6.285714,18.714286,13.571429,13.142857,11.571429,6.857143,19.857143,7.000000,27.875000,60.125000,9.750000,25.875000,13.250000,19.375000,13.500000,7.500000,22.875000,24.875000,58.625000,5.625000,19.000000,12.000000,14.875000,14.000000,7.000000,22.750000,11.250000,0.750000,0.800000,3,14,-11,1963.008325,1607.953020,2013.801340,1727.454953
2,2010,1139,77,1431,59,0,23.250000,47.500000,7.750000,19.000000,6.000000,12.250000,13.000000,6.000000,18.750000,20.500000,56.250000,3.500000,20.500000,11.000000,9.500000,12.250000,6.500000,21.500000,14.000000,26.375000,56.125000,5.375000,15.250000,9.125000,12.625000,13.625000,8.375000,18.625000,21.875000,56.125000,6.500000,19.500000,13.125000,12.500000,14.375000,6.125000,17.250000,7.375000,1.000000,0.800000,5,12,-7,2030.084473,1814.148420,2086.494668,1888.172286
3,2010,1140,99,1196,92,0,27.428571,59.571429,8.000000,19.571429,9.285714,14.142857,9.714286,8.714286,17.714286,25.000000,55.285714,7.714286,20.142857,7.428571,13.428571,16.142857,5.142857,19.571429,12.857143,24.857143,57.428571,5.571429,16.714286,13.285714,12.285714,11.000000,7.000000,12.428571,25.714286,54.714286,7.857143,19.857143,10.000000,13.142857,13.285714,5.428571,17.428571,0.714286,0.750000,0.250000,7,10,-3,1927.385231,1850.299308,1983.810539,1913.918752
4,2010,1242,90,1250,74,0,27.875000,55.625000,7.000000,15.250000,11.250000,16.125000,13.625000,7.750000,17.750000,24.875000,59.250000,6.250000,19.000000,11.625000,12.625000,12.375000,7.500000,20.125000,12.000000,26.500000,52.666667,7.666667,17.333333,9.333333,16.833333,12.333333,7.500000,15.666667,24.333333,55.333333,6.833333,19.833333,9.000000,14.000000,13.666667,6.500000,19.166667,12.333333,1.000000,1.000000,1,16,-15,2164.213384,1434.278730,2205.008246,1593.575625
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1863,2024,1181,64,1301,76,0,29.000000,60.000000,8.500000,23.166667,10.666667,14.833333,8.833333,6.166667,16.500000,25.833333,56.333333,5.666667,15.166667,8.166667,12.333333,9.333333,5.333333,12.000000,8.500000,27.400000,58.400000,6.700000,17.800000,8.800000,12.300000,9.500000,6.500000,13.600000,28.800000,61.700000,7.500000,21.500000,8.200000,14.900000,10.100000,5.800000,16.400000,2.000000,0.333333,0.714286,4,11,-7,2041.214285,1954.783539,2027.939972,1981.341062
1864,2024,1397,66,1345,72,0,25.333333,62.666667,9.166667,27.666667,11.166667,15.500000,9.666667,7.333333,17.666667,24.000000,58.500000,8.666667,27.333333,8.500000,13.500000,12.166667,6.666667,19.333333,6.000000,26.166667,55.666667,7.500000,18.000000,10.000000,18.500000,10.166667,5.166667,16.666667,27.000000,62.666667,6.333333,22.000000,9.666667,12.500000,9.166667,6.833333,23.0

In [8]:
df_test_women_list

,Season,T1_TeamID,T1_Score,T2_TeamID,T2_Score,location,T1_FGM,T1_FGA,T1_FGM3,T1_FGA3,T1_OR,T1_Ast,T1_TO,T1_Stl,T1_PF,T1_opponent_FGM,T1_opponent_FGA,T1_opponent_FGM3,T1_opponent_FGA3,T1_opponent_OR,T1_opponent_Ast,T1_opponent_TO,T1_opponent_Stl,T1_opponent_PF,T1_PointDiff,T2_FGM,T2_FGA,T2_FGM3,T2_FGA3,T2_OR,T2_Ast,T2_TO,T2_Stl,T2_PF,T2_opponent_FGM,T2_opponent_FGA,T2_opponent_FGM3,T2_opponent_FGA3,T2_opponent_OR,T2_opponent_Ast,T2_opponent_TO,T2_opponent_Stl,T2_opponent_PF,T2_PointDiff,T1_win_ratio_14d,T2_win_ratio_14d,T1_seed,T2_seed,Seed_diff,T1_TeamELO,T2_TeamELO,T1_CoachELO,T2_CoachELO
0,2025.0,1101.0,NaN,1102.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1437.981123,1289.363964,1536.933923,1394.093217
1,2025.0,1101.0,NaN,1103.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1437.981123,1692.981797,1536.933923,1821.259533
2,2025.0,1101.0,NaN,1104.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1437.981123,2090.507828,1536.933923,2152.399348
3,2025.0,1101.0,NaN,1105.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1437.981123,1068.425768,1536.933923,1316.636908
4,2025.0,1101.0,NaN,1106.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1437.981123,1131.495622,1536.933923,1383.538287
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131402,2025.0,3477.0,NaN,3479.0,NaN,NaN,21.000000,53.000000,8.750000,22.750000,5.000000,14.0,16.0,6.250000,19.5,22.75,62.750000,4.75,16.750000,13.75,13.0,10.0,9.5,10.750000,-10.250000,21.0,49.50,5.50,19.0,6.00,13.00,20.0,6.0,22.50,26.0,51.50,3.50,12.5,6.5,12.00,19.0,10.0,25.00,-14.5,0.000000,0.0,NaN,NaN,NaN,1135.019364,1241.658681,1612.000528,1612.000528
131403,2025.0,3477.0,NaN,3480.0,NaN,NaN,21.000000,53.000000,8.750000,22.750000,5.000000,14.0,16.0,6.250000,19.5,22.75,62.750000,4.75,16.750000,13.75,13.0,10.0,9.5,10.750000,-10.250000,19.5,57.25,5.25,23.0,14.25,11.25,14.5,8.0,17.75,21.5,50.25,5.25,16.0,5.0,9.75,14.0,7.0,18.75,-3.5,0.000000,0.0,NaN,NaN,NaN,1135.019364,1361.986580,1612.000528,1612.000528
131404,2025.0,3478.0,NaN,3479.0,NaN,NaN,21.166667,53.833333,6.333333,17.833333,9.333333,9.5,11.0,6.166667,11.0,23.50,57.166667,8.00,23.833333,10.50,13.0,12.0,5.5,15.666667,2.166667,21.0,49.50,5.50,19.0,6.00,13.00,20.0,6.0,22.50,26.0,51.50,3.50,12.5,6.5,12.00,19.0,10.0,25.00,-14.5,0.333333,0.0,NaN,NaN,NaN,1209.409031,1241.658681,1612.000528,1612.000528
131405,2025.0,3478.0,NaN,3480.0,NaN,NaN,21.166667,53.833333,6.333333,17.833333,9.333333,9.5,11.0,6.166667,11.0,23.50,57.166667,8.00,23.833333,10.50,13.0,12.0,5.5,15.666667,2.166667,19.5,57.25,5.25,23.0,14.25,11.25,14.5,8.0,17.75,21.5,50.25,5.25,16.0,5.0,9.75,14.0,7.0,18.75,-3.5,0.333333,0.0,NaN,NaN,NaN,1209.409031,1361.986580,1612.000528,1612.000528


In [9]:
df_test_men_list

,Season,T1_TeamID,T1_Score,T2_TeamID,T2_Score,location,T1_FGM,T1_FGA,T1_FGM3,T1_FGA3,T1_OR,T1_Ast,T1_TO,T1_Stl,T1_PF,T1_opponent_FGM,T1_opponent_FGA,T1_opponent_FGM3,T1_opponent_FGA3,T1_opponent_OR,T1_opponent_Ast,T1_opponent_TO,T1_opponent_Stl,T1_opponent_PF,T1_PointDiff,T2_FGM,T2_FGA,T2_FGM3,T2_FGA3,T2_OR,T2_Ast,T2_TO,T2_Stl,T2_PF,T2_opponent_FGM,T2_opponent_FGA,T2_opponent_FGM3,T2_opponent_FGA3,T2_opponent_OR,T2_opponent_Ast,T2_opponent_TO,T2_opponent_Stl,T2_opponent_PF,T2_PointDiff,T1_win_ratio_14d,T2_win_ratio_14d,T1_seed,T2_seed,Seed_diff,T1_TeamELO,T2_TeamELO,T1_CoachELO,T2_CoachELO
0,2025.0,1101.0,NaN,1102.0,NaN,NaN,25.333333,61.0,3.666667,14.333333,10.333333,15.0,15.5,8.5,23.0,24.666667,55.5,4.166667,15.5,10.0,12.166667,16.333333,7.0,17.0,-2.333333,21.000000,52.166667,8.833333,28.166667,5.166667,14.666667,13.833333,6.000000,18.333333,28.000000,56.833333,7.166667,21.666667,7.833333,14.333333,9.500000,8.166667,15.166667,-18.666667,0.333333,0.0,NaN,NaN,NaN,1437.981123,1289.363964,1536.933923,1394.093217
1,2025.0,1101.0,NaN,1103.0,NaN,NaN,25.333333,61.0,3.666667,14.333333,10.333333,15.0,15.5,8.5,23.0,24.666667,55.5,4.166667,15.5,10.0,12.166667,16.333333,7.0,17.0,-2.333333,30.875000,63.000000,9.750000,26.250000,10.625000,15.875000,9.875000,7.250000,16.875000,28.125000,60.875000,7.250000,21.875000,8.750000,11.375000,10.750000,6.125000,17.250000,7.625000,0.333333,1.0,NaN,13.0,NaN,1437.981123,1692.981797,1536.933923,1821.259533
2,2025.0,1101.0,NaN,1104.0,NaN,NaN,25.333333,61.0,3.666667,14.333333,10.333333,15.0,15.5,8.5,23.0,24.666667,55.5,4.166667,15.5,10.0,12.166667,16.333333,7.0,17.0,-2.333333,32.428571,67.000000,11.428571,30.142857,9.000000,17.285714,10.714286,6.714286,19.428571,30.142857,67.857143,8.285714,25.142857,10.142857,15.428571,11.142857,7.571429,19.571429,7.428571,0.333333,0.5,NaN,2.0,NaN,1437.981123,2090.507828,1536.933923,2152.399348
3,2025.0,1101.0,NaN,1105.0,NaN,NaN,25.333333,61.0,3.666667,14.333333,10.333333,15.0,15.5,8.5,23.0,24.666667,55.5,4.166667,15.5,10.0,12.166667,16.333333,7.0,17.0,-2.333333,20.000000,54.333333,7.166667,22.833333,9.333333,12.000000,13.333333,6.666667,17.166667,25.166667,51.333333,8.000000,19.166667,6.833333,13.666667,12.666667,8.333333,17.166667,-13.666667,0.333333,0.0,NaN,NaN,NaN,1437.981123,1068.425768,1536.933923,1316.636908
4,2025.0,1101.0,NaN,1106.0,NaN,NaN,25.333333,61.0,3.666667,14.333333,10.333333,15.0,15.5,8.5,23.0,24.666667,55.5,4.166667,15.5,10.0,12.166667,16.333333,7.0,17.0,-2.333333,24.375000,62.250000,8.000000,25.250000,10.125000,11.250000,8.250000,9.875000,16.625000,21.750000,52.250000,4.875000,17.750000,8.125000,10.375000,13.625000,4.750000,14.875000,8.125000,0.333333,1.0,NaN,16.0,NaN,1437.981123,1131.495622,1536.933923,1383.538287
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131402,2025.0,3477.0,NaN,3479.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1135.019364,1241.658681,1612.000528,1612.000528
131403,2025.0,3477.0,NaN,3480.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1135.019364,1361.986580,1612.000528,1612.000528
131404,2025.0,3478.0,NaN,3479.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1209.409031,1241.658681,1612.000528,1612.000528
131405,2025.0,3478.0,NaN,3480.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1209.409031,1361.986580,1612.000528,1612.000528


## Save x

In [10]:
df_train_women_list.to_csv("WomenTrain.csv")

In [11]:
df_train_men_list.to_csv("MenTrain.csv")

In [12]:
df_test_women_list.to_csv("WomenTest.csv")

In [13]:
df_test_men_list.to_csv("MenTest.csv")